In [ ]:
from typing import Annotated, Any, Dict, List, Optional
from datetime import datetime
from typing_extensions import TypedDict

from dotenv import load_dotenv
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode
from pydantic import BaseModel, Field
from IPython.display import Image, display

import gradio as gr
import uuid

load_dotenv(override=True)


In [ ]:
class EvaluatorOutput(BaseModel):
    feedback: str = Field(description="Feedback on the assistant's response")
    success_criteria_met: bool = Field(description="Whether the success criteria have been met")
    user_input_needed: bool = Field(
        description="True if more input is needed from the user, or clarifications, or the assistant is stuck"
    )


class State(TypedDict, total=False):
    messages: Annotated[List[Any], add_messages]
    success_criteria: str
    feedback_on_work: Optional[str]
    success_criteria_met: bool
    user_input_needed: bool
    research_brief: str


In [ ]:
MOCK_JIRA: Dict[str, Dict[str, str]] = {
    "PROJ-101": {
        "summary": "Fix login timeout",
        "status": "In Progress",
        "description": "Users see 504 after 30s; spike Redis session store.",
    },
    "PROJ-88": {
        "summary": "Bump LangGraph course dep",
        "status": "To Do",
        "description": "Align notebooks with langgraph 0.2+ checkpointers.",
    },
}

calendar_events: List[Dict[str, str]] = []


@tool
def get_jira_issue(issue_key: str) -> str:
    """Fetch a mock Jira issue by key, e.g. PROJ-101 or PROJ-88."""
    issue = MOCK_JIRA.get(issue_key.strip().upper())
    if not issue:
        return f"No issue found for {issue_key!r}. Valid keys: {', '.join(MOCK_JIRA)}."
    return (
        f"{issue_key.upper()}: {issue['summary']} | status={issue['status']} | "
        f"{issue['description']}"
    )


@tool
def add_calendar_event(title: str, start_iso: str) -> str:
    """Add a mock calendar event. start_iso should be an ISO-8601 datetime string."""
    calendar_events.append({"title": title, "start": start_iso})
    return f"Scheduled '{title}' starting {start_iso}"


jira_calendar_tools = [get_jira_issue, add_calendar_event]


In [ ]:
import os

from langchain_community.utilities import GoogleSerperAPIWrapper

if os.getenv("SERPER_API_KEY"):
    _serper = GoogleSerperAPIWrapper()
else:
    _serper = None


In [ ]:
worker_llm = ChatOpenAI(model="gpt-4o-mini")
worker_llm_with_tools = worker_llm.bind_tools(jira_calendar_tools)

evaluator_llm = ChatOpenAI(model="gpt-4o-mini")
evaluator_llm_with_output = evaluator_llm.with_structured_output(EvaluatorOutput)


In [ ]:
def research(state: State) -> Dict[str, Any]:
    task = ""
    for m in state["messages"]:
        if isinstance(m, HumanMessage):
            task = str(m.content)
            break
    if _serper is None:
        brief = "[Set SERPER_API_KEY in .env — see https://serper.dev]"
    else:
        brief = _serper.run(task[:500] if task else "software development best practices")
    return {"research_brief": brief}


def worker(state: State) -> Dict[str, Any]:
    research_brief = state.get("research_brief") or "(no web context)"
    system_message = f"""You are Jira Dev Buddy: help the user plan dev work using mock Jira and a mock calendar.
You may ONLY use the provided tools (get_jira_issue, add_calendar_event). Do not claim to browse the web; web context is already summarized below.
Current date/time: {datetime.now().isoformat(timespec="seconds")}

Web research snippet (may be short or noisy):
{research_brief}

Success criteria:
{state["success_criteria"]}

Reply with a question for the user if you need clarification (start with Question:).
If finished, give the final answer with no question.
"""

    if state.get("feedback_on_work"):
        system_message += (
            "\nPreviously your answer failed evaluation. Feedback: "
            f"{state['feedback_on_work']}\n"
            "Revise to satisfy the success criteria or ask a clear question."
        )

    found_system_message = False
    messages = state["messages"]
    for message in messages:
        if isinstance(message, SystemMessage):
            message.content = system_message
            found_system_message = True
    if not found_system_message:
        messages = [SystemMessage(content=system_message)] + messages

    response = worker_llm_with_tools.invoke(messages)
    return {"messages": [response]}


def worker_router(state: State) -> str:
    last_message = state["messages"][-1]
    if hasattr(last_message, "tool_calls") and last_message.tool_calls:
        return "tools"
    return "evaluator"


def format_conversation(messages: List[Any]) -> str:
    conversation = "Conversation history:\n\n"
    for message in messages:
        if isinstance(message, HumanMessage):
            conversation += f"User: {message.content}\n"
        elif isinstance(message, AIMessage):
            text = message.content or "[Tools use]"
            conversation += f"Assistant: {text}\n"
    return conversation


def evaluator(state: State) -> Dict[str, Any]:
    last_response = state["messages"][-1].content

    system_message = """You are an evaluator. Decide if the assistant met the success criteria and if user input is needed."""
    user_message = f"""Conversation:
{format_conversation(state["messages"])}

Success criteria:
{state["success_criteria"]}

Final assistant response to grade:
{last_response}
"""
    if state.get("feedback_on_work"):
        user_message += (
            f"\nPrior feedback you gave: {state['feedback_on_work']}\n"
            "If the assistant repeats mistakes, set user_input_needed true."
        )

    eval_result = evaluator_llm_with_output.invoke(
        [SystemMessage(content=system_message), HumanMessage(content=user_message)]
    )
    return {
        "messages": [
            {
                "role": "assistant",
                "content": f"Evaluator feedback: {eval_result.feedback}",
            }
        ],
        "feedback_on_work": eval_result.feedback,
        "success_criteria_met": eval_result.success_criteria_met,
        "user_input_needed": eval_result.user_input_needed,
    }


def route_based_on_evaluation(state: State) -> str:
    if state.get("success_criteria_met") or state.get("user_input_needed"):
        return "END"
    return "worker"


In [ ]:
memory = MemorySaver()
graph_builder = StateGraph(State)

graph_builder.add_node("research", research)
graph_builder.add_node("worker", worker)
graph_builder.add_node("tools", ToolNode(tools=jira_calendar_tools))
graph_builder.add_node("evaluator", evaluator)

graph_builder.add_edge(START, "research")
graph_builder.add_edge("research", "worker")
graph_builder.add_conditional_edges(
    "worker",
    worker_router,
    {"tools": "tools", "evaluator": "evaluator"},
)
graph_builder.add_edge("tools", "worker")
graph_builder.add_conditional_edges(
    "evaluator",
    route_based_on_evaluation,
    {"worker": "worker", "END": END},
)

graph = graph_builder.compile(checkpointer=memory)


In [ ]:
# PNG needs network (mermaid.ink). Text diagram works offline:
display(Image(graph.get_graph().draw_mermaid_png()))


In [ ]:
def make_thread_id() -> str:
    return str(uuid.uuid4())


def process_message(message, success_criteria, history, thread):
    config = {"configurable": {"thread_id": thread}}
    state = {
        "messages": [HumanMessage(content=message)],
        "success_criteria": success_criteria
        or "Response should cite relevant Jira fields and be actionable.",
        "feedback_on_work": None,
        "success_criteria_met": False,
        "user_input_needed": False,
        "research_brief": "",
    }
    result = graph.invoke(state, config=config)
    user = {"role": "user", "content": message}
    reply = {"role": "assistant", "content": result["messages"][-2].content}
    feedback = {"role": "assistant", "content": result["messages"][-1].content}
    return history + [user, reply, feedback]


def reset():
    calendar_events.clear()
    return "", "", None, make_thread_id()


with gr.Blocks(theme=gr.themes.Default(primary_hue="emerald")) as demo:
    gr.Markdown("## Jira Dev Buddy")
    thread = gr.State(make_thread_id())
    with gr.Row():
        chatbot = gr.Chatbot(label="Chat", height=360, type="messages")
    with gr.Group():
        with gr.Row():
            msg = gr.Textbox(show_label=False, placeholder="Dev task for the buddy")
        with gr.Row():
            criteria = gr.Textbox(show_label=False, placeholder="Success criteria")
    with gr.Row():
        reset_btn = gr.Button("Reset", variant="stop")
        go_btn = gr.Button("Go!", variant="primary")
    msg.submit(process_message, [msg, criteria, chatbot, thread], [chatbot])
    criteria.submit(process_message, [msg, criteria, chatbot, thread], [chatbot])
    go_btn.click(process_message, [msg, criteria, chatbot, thread], [chatbot])
    reset_btn.click(reset, [], [msg, criteria, chatbot, thread])

demo.launch()
